In [1]:
from modules import Train,HeadClassifierCLIPModel,CLIPExtractor,CLIPCollateFunction,CreationClipDataset,CreationProcessedDataset,creation_dataframe
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor,CLIPImageProcessor,CLIPTokenizerFast,CLIPModel
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.nn.modules.loss import BCEWithLogitsLoss,CrossEntropyLoss
import torch
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


In [2]:
#Creation of the dataframes
train_df=creation_dataframe("../data/train.jsonl")

val_df=creation_dataframe("../data/dev.jsonl")

In [3]:
#Creation of the first Clip Datasets to get the texts and images embeddings through the pretrained clip model
train_clip_dataset=CreationClipDataset(train_df)
val_clip_dataset=CreationClipDataset(val_df)

In [4]:
#Initialisation of Clip Processors
#text_processor=CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32")
#image_processor=CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")
processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
#Initialisation of the clip model, the device, and the batch size
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
batch_size=32

In [6]:
clip_collate_object=CLIPCollateFunction(processor)

In [7]:
#Creation of the first Clip Dataloaders to get the texts and images embeddings through the pretrained clip model
train_clip_dataloader=DataLoader(train_clip_dataset,batch_size=batch_size,shuffle=False,collate_fn=clip_collate_object.collate_fn)
val_clip_dataloader=DataLoader(val_clip_dataset,batch_size=batch_size,shuffle=False,collate_fn=clip_collate_object.collate_fn)

In [8]:
clip_extractor=CLIPExtractor(clip_model,device)

In [9]:
#Extraction of the pretrained CLIP embeddings (texts embeddings, images embeddings, and similarity scores)
final_train_data=clip_extractor.get_embeddings(train_clip_dataloader,"train","./modules/clip_embeddings")
final_val_data=clip_extractor.get_embeddings(val_clip_dataloader,"val","./modules/clip_embeddings")

train CLIP embeddings saved in ./modules/clip_embeddings
val CLIP embeddings saved in ./modules/clip_embeddings


In [10]:
train_data=torch.load("./modules/clip_embeddings/train_clip_embeddings.pt")
val_data=torch.load("./modules/clip_embeddings/val_clip_embeddings.pt")

In [ ]:
#Creation of final datasets and dataloaders for the training
train_dataset=CreationProcessedDataset(train_data)
val_dataset=CreationProcessedDataset(val_data)
train_dataloader=DataLoader(train_dataset,batch_size=32,shuffle=True,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=32,shuffle=True,drop_last=True)

In [12]:
model=HeadClassifierCLIPModel(fc_layer_sizes=[512])

In [13]:
#Use of the class weights to compensate imabalances of the dataset and make more accurate predictions

class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight,dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [ ]:
#Training hyperparameters
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
num_warmup_steps=int(0.1*n_steps)
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=1e-4)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_warmup_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)

In [15]:
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)

In [17]:
trainer.run_training(train_dataloader,val_dataloader,"./modules/train_savings")

2026-03-11 14:06:33.579 | INFO     | modules.train:run_training:106 - Epoch 0 :
2026-03-11 14:06:35.585 | INFO     | modules.train:run_training:169 - Epoch 0: Train Loss = 0.7184474712475798
2026-03-11 14:06:35.585 | INFO     | modules.train:run_training:170 - Epoch 0: Train Accuracy = 0.5481176470588235
2026-03-11 14:06:35.585 | INFO     | modules.train:run_training:171 - Epoch 0: Train F1 = 0.3763597986686151
2026-03-11 14:06:35.585 | INFO     | modules.train:run_training:173 - Epoch 0: Validation Loss = 0.7207121625542641
2026-03-11 14:06:35.585 | INFO     | modules.train:run_training:174 - Epoch 0: Validation Accuracy = 0.54
2026-03-11 14:06:35.585 | INFO     | modules.train:run_training:175 - Epoch 0: Validation F1 = 0.4041450777202072
2026-03-11 14:06:35.602 | INFO     | modules.train:run_training:106 - Epoch 1 :
2026-03-11 14:06:37.132 | INFO     | modules.train:run_training:169 - Epoch 1: Train Loss = 0.6483002685962763
2026-03-11 14:06:37.132 | INFO     | modules.train:run_tra

In [18]:
torch.load("./modules/train_savings/epoch_performances.pt")

{'epoch_train_losses': tensor([0.7184, 0.6483, 0.5764, 0.5310], dtype=torch.float64),
 'epoch_train_f1': tensor([0.3764, 0.5341, 0.6270, 0.6661], dtype=torch.float64),
 'epoch_train_accuracies': tensor([0.5481, 0.6220, 0.7056, 0.7366], dtype=torch.float64),
 'epoch_val_losses': tensor([0.7207, 0.6634, 0.7083, 0.7100], dtype=torch.float64),
 'epoch_val_f1': tensor([0.4041, 0.6004, 0.5298, 0.5747], dtype=torch.float64),
 'epoch_val_accuracies': tensor([0.5400, 0.6220, 0.6060, 0.6240], dtype=torch.float64)}